In [1]:
pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
clinical_match.py: compare the slides in an Athena folder with the clinical Excel file.

Writes clinical_check.xlsx with two sheets:

  Missing slides    DeIDs that are in the Excel but have NO slide in the Athena folder
  Missing data      slides whose clinical data is incomplete:
                      DeID | Slide | Missing   (the blank columns, or "No clinical data in Excel")

The same two lists are printed on screen. Read-only: nothing on Athena or in the Excel changes.

Run on HiPerGator (in the folder that has common.py):
    module load conda; source activate SP_DataManagement
    pip install openpyxl            # once
    python clinical_match.py
"""
import os
import re

import pandas as pd

# ============================================================ settings
EXCEL_PATH = "/orange/pinaki.sarder/Davy_Jones_Locker/Transplant_R01/MVI_Dual_Stained(Acute)/MVI_DeID_0260210.xlsx"   # <- your path on HiPerGator
HEADER_ROW = 1           # column names are on the 2nd row of the MVI Excel
EXCEL_ID_COLUMN = "DeID"

# columns that must be filled in for a slide to count as "complete"
CHECK_COLUMNS = ["Biopsy (Days post-transplant)", "AMR", "TCMR",
                 "g", "i", "t", "v", "ptc", "ti", "ci", "ct", "cv", "ah", "c4d"]
# True = also check the DSA columns (often blank simply because no DSA test was done)
CHECK_DSA = False

INSTANCE = "athena"      # athena | devathena
FOLDER_IDS = [
    # "FOLDER_IDS",
]

RECURSIVE = True

OUTPUT_XLSX = "clinical_check.xlsx"

# finds "JHH_DS_<number>" anywhere in a name: "JHH_DS_1 CD34LCA.svs" -> JHH_DS_1
ID_PATTERN = re.compile(r"JHH[\s_-]*DS[\s_-]*0*(\d+)", re.IGNORECASE)

# readable names for the DSA columns (pandas names repeated headers "Class I", "Class I.1", ...)
DSA_NAMES = {
    "Class I": "DSA pre-biopsy Class I", "Class II": "DSA pre-biopsy Class II",
    "Strength": "DSA pre-biopsy Strength",
    "Class I.1": "DSA at biopsy Class I", "Class II.1": "DSA at biopsy Class II",
    "Strength.1": "DSA at biopsy Strength",
    "Class I.2": "DSA post-biopsy Class I", "Class II.2": "DSA post-biopsy Class II",
    "Strength.2": "DSA post-biopsy Strength",
}


 
 

In [3]:
# ============================================================ helpers
def deid(text):
    m = ID_PATTERN.search(str(text))
    return f"JHH_DS_{int(m.group(1))}" if m else None


def number(d):
    return int(d.rsplit("_", 1)[-1])


def connect():
    import girder_client
    url = {"athena": "https://athena.rc.ufl.edu/api/v1/",
           "devathena": "https://devathena.rc.ufl.edu/api/v1/"}[INSTANCE]
    key = None
    try:                                   # same credentials as Metadata.py, if common.py is here
        import common
        url, key, _, _, _ = common.urlAndCredentials(INSTANCE)
    except Exception:
        key = os.environ.get("ATHENA_API_KEY" if INSTANCE == "athena" else "DEVATHENA_API_KEY")
    gc = girder_client.GirderClient(apiUrl=url)
    if key:
        gc.authenticate(apiKey=key)
    else:
        gc.authenticate(interactive=True)   # asks for username and password
    return gc


def list_slides(gc):
    """Image items only (.svs, .ndpi, ...); .xml and other files are skipped."""
    out = []

    def walk(fid):
        for it in gc.listItem(fid):
            if "largeImage" in it:
                out.append(it["name"])
        if RECURSIVE:
            for sub in gc.listFolder(fid, parentFolderType="folder"):
                walk(sub["_id"])

    for fid in FOLDER_IDS:
        walk(fid)
    return out

In [4]:
# ============================================================ main
def main():
    if not FOLDER_IDS:
        raise SystemExit("Put the Athena folder ID in FOLDER_IDS first.")

    # ---- Excel: DeID -> its row
    df = pd.read_excel(EXCEL_PATH, header=HEADER_ROW)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns=DSA_NAMES)
    columns = CHECK_COLUMNS + ([c for c in DSA_NAMES.values() if c in df.columns] if CHECK_DSA else [])
    not_found = [c for c in [EXCEL_ID_COLUMN] + columns if c not in df.columns]
    if not_found:
        raise SystemExit(f"Not in the Excel: {not_found}\nColumns found: {list(df.columns)}")
    rows = {}
    for _, r in df.iterrows():
        d = deid(r[EXCEL_ID_COLUMN])
        if d:
            rows.setdefault(d, r)

    # ---- Athena: DeID -> slide name(s)
    slides = {}
    for name in list_slides(connect()):
        d = deid(name)
        if d:
            slides.setdefault(d, []).append(name)

    # ---- 1) DeIDs in the Excel with no slide
    missing_slides = pd.DataFrame({"DeID": sorted((d for d in rows if d not in slides), key=number)})

    # ---- 2) slides with incomplete clinical data
    missing_data = []
    for d in sorted(slides, key=number):
        if d not in rows:
            blanks = "No clinical data in Excel"
        else:
            empty = [c for c in columns if pd.isna(rows[d][c]) or str(rows[d][c]).strip() == ""]
            if not empty:
                continue
            blanks = ", ".join(empty)
        for name in slides[d]:
            missing_data.append({"DeID": d, "Slide": name, "Missing": blanks})
    missing_data = pd.DataFrame(missing_data, columns=["DeID", "Slide", "Missing"])

    with pd.ExcelWriter(OUTPUT_XLSX) as xw:
        missing_slides.to_excel(xw, sheet_name="Missing slides", index=False)
        missing_data.to_excel(xw, sheet_name="Missing data", index=False)

    # ---- screen
    print(f"Excel DeIDs: {len(rows)}   Slides in folder: {sum(map(len, slides.values()))} "
          f"({len(slides)} DeIDs)   Columns checked: {len(columns)}")
    print(f"\n1) MISSING SLIDES: {len(missing_slides)} DeIDs in the Excel have no slide in the folder")
    print("   " + ", ".join(missing_slides["DeID"]) if len(missing_slides) else "   none")
    print(f"\n2) MISSING DATA: {len(missing_data)} slides have incomplete clinical data")
    for _, r in missing_data.iterrows():
        print(f"   {r['DeID']:12} missing: {r['Missing']}")
    if not len(missing_data):
        print("   none")
    print(f"\nSaved: {os.path.abspath(OUTPUT_XLSX)}")


if __name__ == "__main__":
    main()

Login or email:  praneethvempati
Password for praneethvempati:  ········


Excel DeIDs: 280   Slides in folder: 139 (139 DeIDs)   Columns checked: 14

1) MISSING SLIDES: 141 DeIDs in the Excel have no slide in the folder
   JHH_DS_6, JHH_DS_8, JHH_DS_26, JHH_DS_37, JHH_DS_44, JHH_DS_45, JHH_DS_46, JHH_DS_47, JHH_DS_49, JHH_DS_50, JHH_DS_51, JHH_DS_52, JHH_DS_53, JHH_DS_54, JHH_DS_55, JHH_DS_56, JHH_DS_57, JHH_DS_58, JHH_DS_59, JHH_DS_60, JHH_DS_61, JHH_DS_62, JHH_DS_63, JHH_DS_64, JHH_DS_65, JHH_DS_66, JHH_DS_67, JHH_DS_68, JHH_DS_69, JHH_DS_70, JHH_DS_71, JHH_DS_72, JHH_DS_73, JHH_DS_74, JHH_DS_75, JHH_DS_76, JHH_DS_77, JHH_DS_78, JHH_DS_79, JHH_DS_80, JHH_DS_81, JHH_DS_82, JHH_DS_83, JHH_DS_86, JHH_DS_94, JHH_DS_95, JHH_DS_96, JHH_DS_97, JHH_DS_98, JHH_DS_99, JHH_DS_107, JHH_DS_109, JHH_DS_111, JHH_DS_112, JHH_DS_117, JHH_DS_121, JHH_DS_122, JHH_DS_132, JHH_DS_133, JHH_DS_134, JHH_DS_139, JHH_DS_143, JHH_DS_145, JHH_DS_146, JHH_DS_149, JHH_DS_150, JHH_DS_151, JHH_DS_159, JHH_DS_166, JHH_DS_173, JHH_DS_174, JHH_DS_187, JHH_DS_189, JHH_DS_195, JHH_DS_198, JHH